# 探索性数据分析EDA

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns",60)
df_raw=pd.read_csv("../data/credit_data(1).csv",index_col="id")
df_raw.head()

## 初步探索
数据形状、数据格式、
根据真实世界和业务理解重新声明数据角色并进行相应的数据类型转变与数据内存压缩

In [ ]:
print(f"原始数据有{df_raw.shape[0]}行，{df_raw.shape[1]}列")
df_raw.info()

In [ ]:
target_variable=["isDefault"]
continuous_variable=["loanAmnt","interestRate","installment","annualIncome","dti","ficoRangeLow","ficoRangeHigh","revolBal", "revolUtil"]
count_variable=["term","delinquency_2years","openAcc","pubRec", "pubRecBankruptcies", "totalAcc"]
categorical_variable=["grade","subGrade","employmentTitle","homeOwnership","verificationStatus","purpose","postCode","regionCode","policyCode","initialListStatus","applicationType","title"]
anonymous_variable=[f"n{i}" for i in range(15)]
date_variable=["issueDate","earliesCreditLine"]

numerical_variable=continuous_variable+count_variable

In [ ]:
# 将工龄统一转换为可空整数，异常非空值显式报错
employment_length = df_raw["employmentLength"].astype("string").str.strip()
employment_length = employment_length.replace({"10+ years": "10 years", "< 1 year": "0 years"})
employment_years = employment_length.str.extract(r"^(\d+)\s*years?$", expand=False)
invalid_employment_length = employment_length.notna() & employment_years.isna()
if invalid_employment_length.any():
    raise ValueError(
        f"有 {int(invalid_employment_length.sum())} 个工龄值无法识别"
    )

df_clean=df_raw.copy()
df_clean["employmentLength"] = pd.to_numeric(employment_years, errors="raise").astype("Int8")

df_clean["employmentLength"]

In [ ]:
date_formats = {
    "issueDate": "%Y-%m-%d",
    "earliesCreditLine": "%b-%Y",
}

def type_transformation(col_name, series):
    if col_name in continuous_variable:
        return pd.to_numeric(series, errors="coerce").astype(float)

    elif col_name in count_variable or col_name in anonymous_variable:
        return pd.to_numeric(series, errors="coerce").astype("Int64")

    elif col_name in categorical_variable:
        return series.astype("category")

    elif col_name in date_variable:
        parsed = pd.to_datetime(
            series,
            format=date_formats[col_name],
            errors="coerce",
        )

        invalid = series.notna() & parsed.isna()

        if invalid.any():
            raise ValueError(
                f"{col_name} 有 {int(invalid.sum())} 个非空日期解析失败"
            )

        return parsed

    return series


for col_name in df_clean.columns:
    df_clean[col_name] = type_transformation(
        col_name,
        df_clean[col_name],
    )

In [ ]:
print(f"内存占用{(df_clean.memory_usage(deep=True).sum()/1024**2):.2f}MB")

## 重复值

In [ ]:
duplication=df_clean.duplicated().sum()
print(f"重复值有{duplication}个")

## 缺失值
1. 可视化确实列
2. 确认缺失相同的组
3. 按年份排序

In [ ]:
#整体缺失
missing_percentage=df_clean.isna().mean().sort_values(ascending=False)
# f,ax=plt.subplots(figsize=(12,6))
# sns.barplot(missing_percentage[missing_percentage>0])
# plt.title("missing percentage(%)")
# plt.xticks(rotation=60,fontsize=7)
# plt.show

In [ ]:
missing_col=missing_percentage[missing_percentage>0].index

In [ ]:
missing_n4=df_clean["n4"].isna()
missing_n10=df_clean["n10"].isna()
if missing_n4.equals(missing_n10):
    print("n4 & n10 为相同的缺失值")

missing_n1=df_clean["n1"].isna()
missing_col=[]
for col in df_clean.columns:
    if df_clean[col].isna().equals(missing_n1):
        missing_col.append(col)
print(f"与 n1 为相同的缺失值的列: {missing_col}")

各种缺失变量中，缺失组违约率较高的有employmentLength和revolUtil，后面要着重考察

In [ ]:
# n_cols=3
# n_rows=(len(missing_col)+n_cols-1)//n_cols

# f,ax=plt.subplots(n_rows,n_cols,figsize=(15,4*n_rows))
# ax=ax.flatten()
# for i,col in enumerate(missing_col):

#     missing_status=pd.Series(np.where(df_clean[col].isna(),"missing","not missing"),index=df_clean.index)
#     s=df_clean.groupby(missing_status)["isDefault"].agg(["mean","sum","size"])
#     sns.barplot(data=s,x=s.index,y="mean",ax=ax[i])
#     ax[i].set_title(f"{col} with Default percentage")
# for j in range(len(missing_col),len(ax)):
#     ax[j].axis("off")



观察缺失值与年份的关系，得知匿名变量在2007-2011年完全缺失，应该是数据库录入的原因，因而后续n组变量缺失值可以忽略

In [ ]:
df_clean["issueYear"] = df_clean["issueDate"].dt.year
df_clean["earliesCreditYear"] = df_clean["earliesCreditLine"].dt.year
missing_time_table = df_clean.groupby("issueYear")[missing_col].agg(lambda x: x.isna().mean())
missing_time_table = missing_time_table.reindex(range(2007, 2019))
print(missing_time_table)

### 查看变量分布并纠正异常值
分类别检查，先describe查看最大最小值
1. 连续型变量-检查无限大或无限小
2. 计数行看是否小于0

In [ ]:
np.isinf(df_clean[numerical_variable]).sum()

In [ ]:
df_clean[continuous_variable].describe()

In [ ]:
df_clean.loc[df_clean["dti"]<0,"dti"]=np.nan
df_clean.loc[df_clean["dti"]==999,"dti"]=np.nan

发现异常值:dti<0和dti=999,替换为缺失值

In [ ]:
# n_cols=4
# n_rows=(len(count_variable)+n_cols-1)//n_cols

# f,ax=plt.subplots(n_rows,n_cols,figsize=(15,4*n_rows))

# ax=ax.flatten()
# for i,col in enumerate(count_variable):
#     sns.histplot(df_clean[col],ax=ax[i],bins=20)

# for j in range(len(count_variable),len(ax)):
#     ax[j].axis("off")

# plt.tight_layout()
# plt.show()


In [ ]:
for col in categorical_variable:
    print(f"{col}有{df_clean[col].nunique()}个取值\n")

ordinal_variable=["grade","subGrade"]
low_cardinality=["homeOwnership","verificationStatus","purpose","initialListStatus","applicationType"]
high_cardinality=["employmentTitle","postCode","regionCode","title"]

In [ ]:
# n_cols=4
# n_rows=(len(anonymous_variable)+n_cols-1)//n_cols

# f,ax=plt.subplots(n_rows,n_cols,figsize=(15,4*n_rows))

# ax=ax.flatten()
# for i,col in enumerate(anonymous_variable):
#     sns.histplot(df_clean[col],ax=ax[i],bins=20)

# for j in range(len(anonymous_variable),len(ax)):
#     ax[j].axis("off")

# plt.tight_layout()
# plt.show()

## 目标样本分类

In [ ]:
positive_target=df_clean[df_clean["isDefault"]==1]
negative_target=df_clean[df_clean["isDefault"]==0]
print(f"目标样本占比{(positive_target.shape[0]/df_clean.shape[0])*100:.2f}%")

In [ ]:
sns.barplot(df_clean["isDefault"].value_counts())

## 各种变量与目标的关系
1. 类别变量-分组
2. 连续变量-等概率分箱
3. 计数变量-低基数看直方图，高技术分箱
4. 时间
5. 匿名-直方图

In [ ]:
# f,ax=plt.subplots(1,2,figsize=(12,5))
# sns.lineplot(df_clean.groupby("grade")["isDefault"].mean(),ax=ax[0])
# sns.lineplot(df_clean.groupby("subGrade")["isDefault"].mean(),ax=ax[1])

In [ ]:
# n_cols=3
# n_rows=(len(low_cardinality)+n_cols-1)//n_cols

# f,ax=plt.subplots(n_rows,n_cols,figsize=(15,4*n_rows))

# ax=ax.flatten()
# for i,col in enumerate(low_cardinality):

#   # 1. 动态使用 col 分组，并计算
#   result = df_clean.groupby(col)["isDefault"].agg(["mean", "sum", "size"])

#   # 2. 显式展示结果
#   sns.barplot(data=result,x=col,y="mean",ax=ax[i])

# for j in range(len(low_cardinality),len(ax)):
#     ax[j].axis("off")

违约率和等级、确认状态、申请类型有关，与房屋产权、申请目的、初始状态关系较弱

In [ ]:
# n_cols=3
# n_rows=(len(high_cardinality)+n_cols-1)//n_cols

# f,ax=plt.subplots(n_rows,n_cols,figsize=(15,4*n_rows))
# ax = ax.flatten()

# for i, col in enumerate(high_cardinality):
#   df_clean[col]=pd.to_numeric(df_clean[col],errors="coerce")
#   sns.kdeplot(
#      data=df_clean,
#      x=col,
#      hue="isDefault",
#      ax=ax[i]     
#   )

# for j in range(len(high_cardinality),len(ax)):
#   ax[j].axis("off")


# plt.tight_layout()
# plt.show()

### 匿名变量-看相关系数

In [ ]:
# sns.heatmap(df_clean[anonymous_variable+target_variable].corr())

n1-n10(除n6)是比较好的变量

### 计数变量-kde

In [ ]:
# n_cols=4
# n_rows=(len(count_variable)+n_cols-1)//n_cols

# f,axes=plt.subplots(n_rows,n_cols,figsize=(15,4*n_rows))
# axes = axes.flatten()

# for i, col in enumerate(count_variable):
#   sns.kdeplot(
#       data=df_clean,
#       x=col,
#       hue="isDefault",  # 核心：按 isDefault 分组画出两组分布曲线
#       fill=True,  # 填充颜色
#       common_norm=False,  # 各自归一化，方便对比形态
#       palette="Set1",  # 颜色主题
#       ax=axes[i],
#   )

#   axes[i].set_title(f"Distribution of {col} by Default")
#   axes[i].grid(True, linestyle="--", alpha=0.3)


# for j in range(len(count_variable),len(axes)):
#     axes[j].axis("off")


# plt.tight_layout()
# plt.show

term和pubRec是影响违约率的重要因素

### 连续型变量

In [ ]:
# n_cols=3
# n_rows=(len(continuous_variable)+n_cols-1)//n_cols

# f,ax=plt.subplots(n_rows,n_cols,figsize=(15,10*n_rows))
# ax=ax.flatten()

# bin_tables={}
# for i,col in enumerate(continuous_variable):
#     bins=pd.qcut(df_clean[col],q=10,duplicates="drop")
#     summary=df_clean.groupby(bins,observed=True)["isDefault"].agg(["mean","sum","size"]).reset_index()
#     summary["bin_str"] = summary[col].astype(str)
#     bin_tables[col] = summary

#     sns.lineplot(data=summary,x="bin_str",y="mean",ax=ax[i])
#     ax[i].set_ylabel("isDefault")
#     ax[i].set_xlabel(summary[col])
#     ax[i].set_xticklabels(summary[col],rotation=90,ha="right",fontsize=7)

# for j in range(len(continuous_variable),len(ax)):
#     ax[j].axis("off") 


# plt.tight_layout()
# plt.show()



### 时间变量

In [ ]:
# sns.lineplot(data=df_clean,x="issueYear",y="isDefault")

违约率与时间的关系变动较大

## 检查

In [ ]:
assert df_clean.index.is_unique, "存在重复贷款 id"
assert df_clean.index.notna().all(), "贷款 id 存在缺失"

assert df_clean["isDefault"].notna().all(), "标签存在缺失"
assert df_clean["isDefault"].isin([0, 1]).all(), "标签不是0/1"

assert df_clean["issueDate"].notna().all(), "放款日期存在缺失"
assert df_clean["earliesCreditLine"].notna().all(), "最早信用日期存在缺失"

df_clean.to_csv(
    "D:/credit/data/data_processing.csv",
    index_label="id",
)